# A/B 测试高级实战：Delta Method 与 CUPED 方差缩减
> **场景**: 我们需要评估一个新的推荐排序策略。为了更精细地评估，我们需要计算**点击率 (CTR)** 这个比例指标 (Ratio Metric)。同时，业务方希望能缩短实验周期，我们需要用到**方差缩减 (CUPED)** 技术。
>
> **目标**: 
> 1. 学会使用 **Delta Method** 准确估算比例指标的方差 (这是 Senior DA 面试必考题)。
> 2. 学会使用 **CUPED** 利用实验前的数据来降低方差，提升检验灵敏度。


---
## Module 0: 环境配置 & 数据生成

真实业务中，CTR 类数据往往是用户级别的聚合日志。我们在这里生成一份模拟记录。每个用户不仅有实验期 (exp) 的曝光/点击数据，还有实验前 (pre) 的数据。


In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# 中文字体配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

# 生成模拟数据
np.random.seed(42)
N = 20000

# 1. 生成实验前数据 (Covariates)
# 曝光量 ~ Poisson(100), CTR ~ 0.05
pre_impressions = np.random.poisson(100, N)
pre_ctr_true = np.clip(np.random.normal(0.05, 0.01, N), 0.01, 0.1)
pre_clicks = np.random.binomial(pre_impressions, pre_ctr_true)

# 2. 分配实验组与对照组
groups = np.random.choice(['control', 'treatment'], N)
is_treat = (groups == 'treatment').astype(int)

# 3. 生成实验期数据
# 用户在实验期的活跃度与实验前高度正相关
exp_impressions = np.random.poisson(pre_impressions) 

# Treatment group 有 0.5% (0.005) 的绝对 CTR 提升
uplift = 0.005
exp_ctr_true = np.clip(pre_ctr_true + is_treat * uplift + np.random.normal(0, 0.005, N), 0.01, 0.15)
exp_clicks = np.random.binomial(exp_impressions, exp_ctr_true)

# 合并为 DataFrame
df = pd.DataFrame({
    'user_id': range(1, N+1),
    'group': groups,
    'pre_impressions': pre_impressions,
    'pre_clicks': pre_clicks,
    'exp_impressions': exp_impressions,
    'exp_clicks': exp_clicks
})

print(f"数据总行数: {len(df)}")
df.head()


数据总行数: 20000


,user_id,group,pre_impressions,pre_clicks,exp_impressions,exp_clicks
0,1,control,96,7,84,3
1,2,control,107,4,117,6
2,3,control,88,3,98,4
3,4,control,103,5,115,3
4,5,control,111,4,114,2


---
## Module 1: 为什么需要 Delta Method？ (Senior 必考)

### 什么是比例指标 (Ratio Metric)?
像 ARPU (平均每用户收入) 这样的指标，分母是**用户数** (也是我们随机分流的基本单位)。
而 **CTR (总点击 / 总曝光)**，它的分母是**曝光数**，这是一个**随机变量** (有些用户活跃曝光200次，有些用户只曝光10次)。

如果你直接算每个用户的 CTR 然后求平均，那是 `Average of Ratios` (用户的平均点击率)，不是大盘的 `Ratio of Averages` (总体点击率)。业务看的永远是大盘总体点击率！

### Delta Method 方差计算公式
对于 \(CTR = \frac{\sum Clicks}{\sum Impressions} = \frac{\overline{X}}{\overline{Y}}\)，其方差近似为：
$$Var\left(\frac{\overline{X}}{\overline{Y}}\right) \approx \frac{1}{n} \frac{\mu_X^2}{\mu_Y^2} \left[ \frac{Var(X)}{\mu_X^2} + \frac{Var(Y)}{\mu_Y^2} - 2\frac{Cov(X,Y)}{\mu_X \mu_Y} \right]$$

> **大白话**: 因为分子(Clicks)和分母(Impressions)是高度相关的，不能简单地用二项分布算方差。必须通过包含协方差 (Covariance) 的 Delta Method 来准确估计。


In [3]:
# 🏋️ 你的代码: 实现 Delta Method 计算方差与 T-stat
# 提示: 计算出控制组和实验组的 true variance, 然后手写 t-stat
def delta_variance(df_tb,fenzi,fenmu):
    x = df_tb[fenzi]
    y = df_tb[fenmu]

    n = len(df_tb)
    mean_x = x.mean()
    mean_y = y.mean()
    var_x = x.var()
    var_y = y.var()
    cov_xy = np.cov(x,y)[0,1] # 

    var_ratio = (mean_x**2 / mean_y**2) * (var_x / mean_x**2 + var_y / mean_y**2 - 2 * cov_xy / (mean_x * mean_y ))

    return(mean_x / mean_y),(var_ratio / n )

control = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

metric_c,var_c = delta_variance(control,'exp_clicks','exp_impressions')
metric_t,var_t = delta_variance(treatment,'exp_clicks','exp_impressions')

diff = metric_t - metric_c
se = np.sqrt(var_c + var_t)
t_stat = diff / se 
p_value = 2 * (1- stats.norm.cdf(abs(t_stat)))

print(f'实验组指标:{metric_t:.4%},方差:{var_t:.4f}')
print(f'控制组指标:{metric_c:.4%},方差:{var_c:.4f}')
print(f'实验组效应:{diff:.4%},P值:{p_value:.4f},{"显著" if p_value < 0.05 else "不显著"}')
print(f't统计值(deltamethod):{t_stat:.4f}')





实验组指标:5.5320%,方差:0.0000
控制组指标:4.9868%,方差:0.0000
实验组效应:0.5452%,P值:0.0000,显著
t统计值(deltamethod):15.5006


In [4]:
# ✅ 参考答案: Delta Method 实操
def cal_delta_variance(df_group, click_col, imp_col):
    x = df_group[click_col]
    y = df_group[imp_col]
    
    n = len(df_group)
    mean_x = x.mean()
    mean_y = y.mean()
    var_x = x.var()
    var_y = y.var()
    cov_xy = np.cov(x, y)[0, 1]
    
    # Delta Method variance for Ratio (mean_x / mean_y)
    var_ratio = (mean_x**2 / mean_y**2) * (var_x / mean_x**2 + var_y / mean_y**2 - 2 * cov_xy / (mean_x * mean_y))
    # Note: the variance of the sample mean ratio is var_ratio / n
    return (mean_x / mean_y), (var_ratio / n)

ctrl = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

ctr_c, var_c = cal_delta_variance(ctrl, 'exp_clicks', 'exp_impressions')
ctr_t, var_t = cal_delta_variance(treat, 'exp_clicks', 'exp_impressions')

# T-statistic
diff = ctr_t - ctr_c
se = np.sqrt(var_c + var_t)
t_stat = diff / se
p_value = 2 * (1 - stats.norm.cdf(abs(t_stat)))

print(f"Control CTR: {ctr_c:.5f}, Variance: {var_c:.8f}")
print(f"Treatment CTR: {ctr_t:.5f}, Variance: {var_t:.8f}")
print(f"Uplift: {diff:.5f}")
print(f"T-statistic (Delta Method): {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")


Control CTR: 0.04987, Variance: 0.00000006
Treatment CTR: 0.05532, Variance: 0.00000006
Uplift: 0.00545
T-statistic (Delta Method): 15.5006
P-value: 0.000000


---
## Module 2: CUPED 方差缩减技术

### 为什么用 CUPED？
方差 (\(Var\)) 越小，T 统计量越大，P-value 越容易显著！
如果我们能找到一个在实验前既有的特征 (Covariate)，它与实验期的核心指标高度相关 (如：用户实验前的点击量)，我们就可以用它来"解释"掉一部分方差。剩下无法解释的方差变小了，实验的灵敏度就提高了！

### CUPED 核心公式
1. **修正后的指标**：
$$Y_{cuped} = Y - \theta (X - E[X])$$
其中 \(Y\) 是实验期指标，\(X\) 是实验前协变量 (Covariate)。

2. **最优的 \(\theta\)**：
$$\theta = \frac{Cov(X, Y)}{Var(X)}$$

这在本质上就是一个**线性回归**的斜率！我们把方差缩减到了原来的 \((1 - \rho^2)\) 倍，相关系数 \(\rho\) 越高，方差缩减越明显。


In [ ]:
# 🏋️ 你的代码: 实现 CUPED 取代原始 clicks 指标，并观察 Variance 变化
# 目标: 将 exp_clicks 进行 CUPED 修正，然后分别计算修正前后的 Variance


In [4]:
# ✅ 参考答案: CUPED 完整模块 (计算 + 五步稳健性检验)
# 我们以用户维度的总点击量 (exp_clicks) 作为核心指标演示 CUPED

# ============================================================
# Part 1: CUPED 核心计算
# ============================================================

y = df['exp_clicks']   # 实验期指标
x = df['pre_clicks']   # 实验前指标 (Covariate)

# Step 1: 计算最优 Theta（必须使用全局数据，防止引入 Bias）
theta = np.cov(x, y)[0, 1] / np.var(x, ddof=1)
print(f"计算所得最优 Theta: {theta:.4f}")
print(f"协变量相关系数 ρ: {np.corrcoef(x, y)[0, 1]:.4f}")

# Step 2: 计算修正后的 CUPED 指标
mean_x = np.mean(x)
df['exp_clicks_cuped'] = df['exp_clicks'] - theta * (df['pre_clicks'] - mean_x)

# ============================================================
# Part 2: 五步稳健性检验（类比因果推断 Balance Check）
# ============================================================

PASS_EMOJI = "✅"
FAIL_EMOJI = "❌"
SEPARATOR = "=" * 50

# --- Check 1: 均值不变性检验 ---
# CUPED 的数学保证: E[Y_cuped] = E[Y]，修正前后均值不变
print(f"\n{SEPARATOR}")
print("🔍 Check 1: 均值不变性检验 (Mean Invariance)")
print(f"{SEPARATOR}")
ALL_PASS = True
for g in ['control', 'treatment']:
    mask = df['group'] == g
    orig_mean = df.loc[mask, 'exp_clicks'].mean()
    cuped_mean = df.loc[mask, 'exp_clicks_cuped'].mean()
    diff = abs(orig_mean - cuped_mean)
    MEAN_TOLERANCE = 1e-6
    status = PASS_EMOJI if diff < MEAN_TOLERANCE else FAIL_EMOJI
    if diff >= MEAN_TOLERANCE:
        ALL_PASS = False
    print(f"  {g}: 原始均值={orig_mean:.6f}, CUPED均值={cuped_mean:.6f}, 差值={diff:.10f} {status}")
print(f"  结论: {'均值不变，修正合法' if ALL_PASS else '⚠️ 均值偏移，检查代码！'}")

# --- Check 2: 方差缩减 > 0 ---
print(f"\n{SEPARATOR}")
print("🔍 Check 2: 方差缩减有效性 (Variance Reduction)")
print(f"{SEPARATOR}")
var_orig = np.var(df['exp_clicks'], ddof=1)
var_cuped = np.var(df['exp_clicks_cuped'], ddof=1)
variance_reduction = 1 - var_cuped / var_orig
status = PASS_EMOJI if variance_reduction > 0 else FAIL_EMOJI
print(f"  原始指标方差: {var_orig:.4f}")
print(f"  CUPED指标方差: {var_cuped:.4f}")
print(f"  方差缩减比例: {variance_reduction:.2%} {status}")
if variance_reduction <= 0:
    print("  ⚠️ 方差未缩减！协变量与实验指标相关性太弱，建议换协变量")

# --- Check 3: 协变量必须是实验前的 (Pre-Experiment Only) ---
print(f"\n{SEPARATOR}")
print("🔍 Check 3: 协变量时间窗口检查 (Pre-Experiment Only)")
print(f"{SEPARATOR}")
# 在真实项目中，这里应该检查协变量的时间戳 < 实验开始时间
# 在模拟数据中，'pre_clicks' 由设计保证是实验前的
print(f"  协变量字段: 'pre_clicks' (实验前点击量)")
print(f"  ✅ 已确认: 协变量严格属于实验前时间窗口，未受实验策略影响")

# --- Check 4: 全局共享 Theta（不分组各算各的）---
print(f"\n{SEPARATOR}")
print("🔍 Check 4: Theta 全局一致性 (Global vs Per-Group)")
print(f"{SEPARATOR}")
theta_global = theta  # 已用全局数据计算
theta_ctrl = np.cov(
    df[df['group']=='control']['pre_clicks'],
    df[df['group']=='control']['exp_clicks']
)[0, 1] / np.var(df[df['group']=='control']['pre_clicks'], ddof=1)
theta_treat = np.cov(
    df[df['group']=='treatment']['pre_clicks'],
    df[df['group']=='treatment']['exp_clicks']
)[0, 1] / np.var(df[df['group']=='treatment']['pre_clicks'], ddof=1)
print(f"  全局 Theta:   {theta_global:.4f} ← 应该使用这个！")
print(f"  对照组 Theta: {theta_ctrl:.4f}")
print(f"  实验组 Theta: {theta_treat:.4f}")
print(f"  ✅ 已确认: 使用全局 Theta，避免实验效应污染系数")

# --- Check 5: Uplift 不变 ---
print(f"\n{SEPARATOR}")
print("🔍 Check 5: 效果量不变性 (Uplift Invariance)")
print(f"{SEPARATOR}")
uplift_orig = (df[df['group']=='treatment']['exp_clicks'].mean() -
               df[df['group']=='control']['exp_clicks'].mean())
uplift_cuped = (df[df['group']=='treatment']['exp_clicks_cuped'].mean() -
                df[df['group']=='control']['exp_clicks_cuped'].mean())
uplift_diff = abs(uplift_orig - uplift_cuped)
UPLIFT_TOLERANCE = 1e-6
status = PASS_EMOJI if uplift_diff < UPLIFT_TOLERANCE else FAIL_EMOJI
print(f"  原始 Uplift: {uplift_orig:.6f}")
print(f"  CUPED Uplift: {uplift_cuped:.6f}")
print(f"  差值: {uplift_diff:.10f} {status}")
print(f"  结论: {'Uplift 不变，CUPED 只缩减方差不改变效果量' if uplift_diff < UPLIFT_TOLERANCE else '⚠️ Uplift 偏移，检查公式！'}")

# ============================================================
# Part 3: 对比 T-test 结果（CUPED 前 vs 后）
# ============================================================
print(f"\n{SEPARATOR}")
print("📊 最终结果: T-test 对比 (CUPED 前 vs 后)")
print(f"{SEPARATOR}")

t_orig, p_orig = stats.ttest_ind(
    df[df['group']=='treatment']['exp_clicks'],
    df[df['group']=='control']['exp_clicks']
)
t_cuped, p_cuped = stats.ttest_ind(
    df[df['group']=='treatment']['exp_clicks_cuped'],
    df[df['group']=='control']['exp_clicks_cuped']
)

print(f"  原始指标: T={t_orig:.4f}, P-value={p_orig:.6f}")
print(f"  CUPED指标: T={t_cuped:.4f}, P-value={p_cuped:.6f}")
print(f"  T 统计量提升: {abs(t_cuped)/abs(t_orig) - 1:.1%}")
if p_cuped < p_orig:
    print("  🎯 CUPED 成功降低 P-value，更容易检测出真实的效应差异！")

计算所得最优 Theta: 0.2068
协变量相关系数 ρ: 0.1945

🔍 Check 1: 均值不变性检验 (Mean Invariance)
  control: 原始均值=4.989438, CUPED均值=4.989994, 差值=0.0005556324 ❌
  treatment: 原始均值=5.525686, CUPED均值=5.525116, 差值=0.0005703832 ❌
  结论: ⚠️ 均值偏移，检查代码！

🔍 Check 2: 方差缩减有效性 (Variance Reduction)
  原始指标方差: 6.7766
  CUPED指标方差: 6.5202
  方差缩减比例: 3.78% ✅

🔍 Check 3: 协变量时间窗口检查 (Pre-Experiment Only)
  协变量字段: 'pre_clicks' (实验前点击量)
  ✅ 已确认: 协变量严格属于实验前时间窗口，未受实验策略影响

🔍 Check 4: Theta 全局一致性 (Global vs Per-Group)
  全局 Theta:   0.2068 ← 应该使用这个！
  对照组 Theta: 0.2057
  实验组 Theta: 0.2077
  ✅ 已确认: 使用全局 Theta，避免实验效应污染系数

🔍 Check 5: 效果量不变性 (Uplift Invariance)
  原始 Uplift: 0.536248
  CUPED Uplift: 0.535122
  差值: 0.0011260156 ❌
  结论: ⚠️ Uplift 偏移，检查公式！

📊 最终结果: T-test 对比 (CUPED 前 vs 后)
  原始指标: T=14.6424, P-value=0.000000
  CUPED指标: T=14.8989, P-value=0.000000
  T 统计量提升: 1.8%
  🎯 CUPED 成功降低 P-value，更容易检测出真实的效应差异！


---
## Module 3: 🎤 面试实战 (Senior Talk Track)

> **面试官**: 我们上线了一个短视频推荐策略，CTR 的提升在 T-test 下 p-value 是 0.08 (未达 0.05 显著)。业务方觉得策略明明挺好的，急着要上线。作为数据分析师，你怎么处理？

**请在下方写出你的核心思考框架与应对策略（结合 CUPED 或样本量/效果大小）：**


### [记录你的回答思路...]
